# Benchmark 2: single-stage compression to 4 fs (1.5 cycles) in a helium-filled MPC

**Paper:** L. Daniault, J. Kaur, G. Gallé, C. Sire, F. Sylla, R. Lopez-Martens, *Sub-2-cycle post-compression of multi-mJ energy Ti:sapphire laser pulses in a gas-filled multi-pass cell*, Opt. Lett. (December 2024), open access at [arXiv:2410.12525](https://arxiv.org/abs/2410.12525).

**Why this case:** it pushes the UPPE as far as an MPC goes, from 35 fs to 4 fs in a single stage, with a nearly octave-spanning spectrum. Self-steepening, the helium dispersion over 83 m and the Kerr lens at P/Pc ≈ 0.74 all matter at once. The paper also measures the output spectrum **versus pressure** (Fig. 2): linear broadening up to ~1200 mbar, saturation as dispersion smooths the spectrum, distortion above 1800 mbar and pulse breakdown from self-steepening above 2200 mbar. That curve is a much stronger test than a single operating point, and the last section of this notebook rebuilds it from several runs.

| | paper | used here |
|---|---|---|
| laser | Ti:Sa, 800 nm, 35 fs, 6.7 mJ into the cell | Gaussian 35 fs (or a measured spectrum in `data/`), 6.7 mJ |
| mirrors | enhanced silver, RoC 1.5 m, 4", ~99 %/bounce, 69 % over 28 passes | R = 0.987 (gives 69 %), flat over 430–1250 nm, GDD = 0 (assumptions) |
| geometry | ~3 m, highly concentric, 28 passes | `MPC(R=1.5, d=2.981, n_passes=28)`; d chosen to give the quoted eigenmode |
| eigenmode | 350 µm diameter at the waist, 4.3 mm on the mirrors | computed below (radii) |
| gas | He, 1800 mbar (scan 0–2200 mbar), **circular polarisation** | He, n₂(1 bar) = 3.5×10⁻²⁵ m²/W × 2/3 (scalar code: circular polarisation enters as n₂,eff = ⅔ n₂) |
| compressor | 8 pairs of PC70 chirped mirrors (−640 fs²) + fused-silica wedges | −640 fs² + the fused-silica thickness that maximises the peak power |

**Measured results to compare with:** 4.0 fs compressed, 3.7 fs Fourier limit, temporal Strehl ratio 90 %, 69 % cell throughput, homogeneity > 99 % within the FWHM and > 97 % within 1/e² on both axes.

**Known limits:** the code has no ionisation (the authors' own simulations found it negligible at these settings; the peak-intensity plot shows how close the run gets), no polarisation dynamics beyond the ⅔ factor, no chirped-mirror GDD oscillations, and no residual input phase (the authors attribute the central peak at 800 nm of their spectrum to it).

In [ ]:
import os
GPU_ID = "0"          # which GPU this notebook uses (restart the kernel to change it); "0" with one GPU
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

In [ ]:
import os, time, json, hashlib
import numpy as np
import matplotlib.pyplot as plt
import thunder4d as th
from thunder4d import diagnostics as dg

FS = 1e-15
plt.rcParams.update({"figure.dpi": 110})
print("thunder4d", th.__version__, "| GPU available:", th.gpu_available())

In [ ]:
# ---------------- run settings ----------------
QUICK = False          # True: 2 passes on a coarse grid, only to check that the notebook runs
DEVICE = "auto"        # "auto" (GPU if CuPy works), "gpu" or "cpu"
CPU_BUDGET_GB = 16     # memory the run may use when it runs on the CPU
PHI_MAX = 0.02         # peak nonlinear phase per step: halve it once to check convergence
MAX_SECONDS = None     # e.g. 3600: stop cleanly after ~1 h, re-run the run cell to continue
RESUME = True          # continue from the checkpoint of an identical run if there is one
CACHE_SIZE = 1         # propagators kept in GPU memory (solver default 4): 1 saves 3 field copies
N_FIELDS = 10 + CACHE_SIZE   # peak number of full-size arrays on the GPU, used to choose the grid

In [ ]:
# ---------------- helpers (same in the three benchmark notebooks) ----------------
def device_budget_gb():
    """Memory the run may use: 80 % of the free GPU memory, or CPU_BUDGET_GB on the CPU."""
    if DEVICE != "cpu" and th.gpu_available():
        import cupy as cp
        free, total = cp.cuda.runtime.memGetInfo()
        print(f"GPU: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
        return 0.8 * free / 1e9
    print(f"CPU run, budget {CPU_BUDGET_GB} GB")
    return CPU_BUDGET_GB


def pick_grid(cell, presets, budget_gb, n_fields=12, **grid_args):
    """Finest preset whose estimated memory (about n_fields x one field) fits in budget_gb."""
    for p in presets:
        g = th.suggest_grid(cell, **grid_args, **p, device="cpu")       # axes only: cheap
        need = n_fields * np.prod(g.shape) * 8 / 1e9                    # complex64
        ok = need <= budget_gb
        print(f"  {p}: {g.Nx}x{g.Ny}x{g.Nt}, ~{need:.1f} GB" + ("  <- used" if ok else ""))
        if ok:
            return th.suggest_grid(cell, **grid_args, **p, device=DEVICE), p
    raise MemoryError("no preset fits the memory budget: add a coarser preset or use a bigger GPU")


def band_mirror(R_in, lam_lo, lam_hi, R_out=0.0, order=80):
    """Intensity reflectivity R(lambda) [lambda in m]: R_in inside the band, smooth edges with their
    50 % points at lam_lo and lam_hi, R_out far outside."""
    lc, hw = 0.5 * (lam_lo + lam_hi), 0.5 * (lam_hi - lam_lo)
    def R(lam):
        u = np.abs((np.asarray(lam, float) - lc) / hw)
        return R_out + (R_in - R_out) * np.exp(-np.log(2) * u ** order)
    return R


def gdd_phase(grid, gdd_fs2=0.0, tod_fs3=0.0):
    """Spectral phase on grid.Omega (FFT order) of a GDD [fs^2] and TOD [fs^3]."""
    W = grid.Omega * FS
    return gdd_fs2 / 2 * W ** 2 + tod_fs3 / 6 * W ** 3


def beam_pixels(Aw, rel=1e-4):
    """(npix, Nt) host array of the pixels that carry the beam (fluence > rel x max)."""
    A = th.to_host(Aw)
    F = np.sum(np.abs(A) ** 2, axis=2)
    return A[F > rel * F.max()]


def _pad(E, up):
    """Zero-pad spectra (FFT order, last axis) by `up` -> time step dt/up."""
    if up == 1:
        return E
    N = E.shape[-1]
    h = (N + 1) // 2
    out = np.zeros(E.shape[:-1] + (N * up,), complex)
    out[..., :h] = E[..., :h]
    out[..., N * up - (N - h):] = E[..., h:]
    return out


def integrated_power(grid, E_pix, phase=None, up=4):
    """Spatially integrated P(t) [W] of E_pix (npix, Nt) after the spectral phase `phase`
    (array on grid.Omega). Returns sorted t [fs] and P [W], time step dt/up."""
    E = E_pix if phase is None else E_pix * np.exp(1j * phase)[None, :]
    A = grid.hfft.ift_t(_pad(E, up)) * up
    P = np.sum(np.abs(A) ** 2, axis=0) * grid.dA
    t = np.fft.fftfreq(grid.Nt * up, 1.0 / (grid.Nt * grid.dt))
    return np.fft.fftshift(t) / FS, np.fft.fftshift(P)


def fourier_limit(grid, E_pix, up=4):
    """Fourier limit of the spatially integrated spectrum (the 'Fourier limit' quoted in papers)."""
    S = np.sum(np.abs(E_pix) ** 2, axis=0)
    return integrated_power(grid, np.sqrt(S)[None, :], None, up)


def fwhm(x, y):
    """FWHM of the main peak (linear interpolation)."""
    y = np.asarray(y, float)
    i = int(np.argmax(y)); h = y[i] / 2
    l, r = i, i
    while l > 0 and y[l] > h: l -= 1
    while r < len(y) - 1 and y[r] > h: r += 1
    xl = np.interp(h, [y[l], y[l + 1]], [x[l], x[l + 1]])
    xr = np.interp(h, [y[r], y[r - 1]], [x[r], x[r - 1]])
    return xr - xl


def span(lam, S, level_db=-20):
    """(lambda_min, lambda_max) [nm] of the outermost points above level_db relative to the peak."""
    ok = np.nonzero(S / S.max() >= 10 ** (level_db / 10))[0]
    return lam[ok[0]] * 1e9, lam[ok[-1]] * 1e9


def homogeneity_1d(S, rel=np.exp(-2)):
    """Spectral homogeneity as defined by Weitenberg et al. (Opt. Express 25, 20502, 2017) and used in
    the papers: V(x) = (int sqrt(S S_ref))^2 / (int S int S_ref), S_ref = spectrum at the intensity
    maximum. S[x, lambda]. Returns V(x) and its intensity-weighted mean inside the 1/e^2 width."""
    prof = S.sum(axis=1)
    S0 = S[np.argmax(prof)]
    V = np.sum(np.sqrt(S * S0[None, :]), axis=1) ** 2 / np.maximum(S.sum(axis=1) * S0.sum(), 1e-300)
    inside = prof > rel * prof.max()
    return V, float(np.sum(V[inside] * prof[inside]) / np.sum(prof[inside]))


def load_digitized(path):
    """Optional measured curve digitised from a paper figure: CSV with 2 columns (x, y)."""
    if not os.path.exists(path):
        print(f"(no {path}: digitise the figure, e.g. with WebPlotDigitizer, to overlay the measurement)")
        return None
    d = np.loadtxt(path, delimiter=",")
    d = d[np.argsort(d[:, 0])]
    return d[:, 0], d[:, 1] / d[:, 1].max()


def table(rows):
    """rows: (quantity, simulation, paper) -> aligned text table."""
    print(f"{'quantity':42s}{'simulation':>16s}{'paper':>22s}")
    print("-" * 80)
    for q, s, p in rows:
        print(f"{q:42s}{s:>16s}{p:>22s}")


def run_tag(params, grid):
    """Short fingerprint of the parameters and grid, used to name the checkpoint."""
    s = json.dumps({k: str(v) for k, v in sorted(params.items())}) + str(grid.shape) + str(PHI_MAX) + str(QUICK)
    return hashlib.md5(s.encode()).hexdigest()[:8]

## 1. Parameters from the paper

In [ ]:
P = dict(
    lambda0=800e-9,
    energy=6.7e-3,           # paper: effective energy sent into the MPC
    fwhm=35e-15,             # paper: 35 fs; residual input GDD and TOD were cancelled
    input_spectrum_csv="data/daniault2024_input_spectrum.csv",   # optional measured laser spectrum (nm, I)
    pressure_bar=1.8,        # paper: optimum 1800 mbar. Pressure scan: 0.4 ... 2.2 bar
    n2_he_1bar=3.5e-25,      # literature value for linear polarisation (uncertain by ~15 %)
    circular=True,           # paper: circular polarisation -> n2_eff = 2/3 n2
    R=1.5, d=2.981,          # paper: RoC 1.5 m, ~3 m long; d reproduces the quoted eigenmode
    n_passes=28,             # paper: throughput characterised for 28 passes (38 possible)
    mirror_R=0.987,          # paper: ~99 % per bounce, 69 % over 28 passes -> 0.987
    band=(430e-9, 1250e-9),  # assumption: enhanced silver flat over the broadened spectrum
    R_out=0.9,               # assumption: reflectivity far outside that band
    mirror_gdd=0.0,          # assumption: not given in the paper
    cm_gdd=-640.0,           # paper: 8 pairs of PC70 chirped mirrors
    fs_scan_mm=(-2.0, 6.0),  # range of fused-silica thickness tried in the compressor
    lambda_min=450e-9, lambda_max=1400e-9,   # spectral window of the grid
    window_fwhm=120e-15,     # sets the time window (x time_window_factor): the output is chirped
    injection="linear",      # "linear" (eigenmode quoted in the paper) or "nonlinear" (Kerr-matched)
)
REF = dict(w0_um=175, wm_um=2150, T=0.69, tl_fs=3.7, comp_fs=4.0, strehl=0.90,
           V="> 99 % FWHM, > 97 % 1/e^2")
RECORD_EVERY = 2
SCAN_DIR = "runs_bench_daniault"
RUN_DIR = os.path.join(SCAN_DIR, f"p{int(round(P['pressure_bar'] * 1000))}mbar")
os.makedirs(RUN_DIR, exist_ok=True)
print("run directory:", RUN_DIR)

## 2. Gas, cell and eigenmode

In [ ]:
n2_1bar = P["n2_he_1bar"] * (2 / 3 if P["circular"] else 1.0)
gas = th.Material("He", P["pressure_bar"], n2_1bar=n2_1bar)
cell = th.MPC(R=P["R"], d=P["d"], medium=gas, n_passes=2 if QUICK else P["n_passes"],
              mirror_gdd_fs2=P["mirror_gdd"],
              mirror_reflectivity=band_mirror(P["mirror_R"], *P["band"], R_out=P["R_out"]))
lin = cell.linear_mode(P["lambda0"])
print(cell.report())
print(f"linear eigenmode: waist {lin['w0']*1e6:.0f} um (paper {REF['w0_um']}), "
      f"mirror {lin['w_mirror']*1e6:.0f} um (paper ~{REF['wm_um']})  [1/e^2 radii]")
print(f"Pc = {gas.critical_power(P['lambda0'])/1e9:.0f} GW, GVD = {gas.gvd(P['lambda0']):.2f} fs^2/m, "
      f"GDD of the gas over {P['n_passes']} passes: {gas.gvd(P['lambda0']) * P['d'] * P['n_passes']:.0f} fs^2")

## 3. Grid
The grid's carrier is put at the centre (in frequency) of the 450–1400 nm window, about 681 nm, instead of 800 nm. The frequency window is symmetric about the carrier, so with an 800 nm carrier it would span 451–3600 nm and waste about a third of the time points. The physics does not depend on the carrier: the UPPE is solved for every frequency of the window, and the frame drift between the group velocities at 681 and 800 nm in helium is below 1 fs over the whole cell. Only three places need care, and they are handled below: the pulse spectrum is centred at 800 nm explicitly, beam sizes given at 800 nm are converted to the carrier, and spectral phases (GDD, glass) are expanded about 800 nm.

The presets are sized for a 16 GB card: about 11 field copies must fit in 80 % of the free memory.

In [ ]:
LAM_GRID = 2 / (1 / P["lambda_min"] + 1 / P["lambda_max"])      # carrier at the centre of the window
PRESETS = ([dict(box_factor=4, points_per_waist=3, time_window_factor=4)] if QUICK else
           [dict(box_factor=6, points_per_waist=6, time_window_factor=7),      # ~22 GB
            dict(box_factor=5.5, points_per_waist=5.5, time_window_factor=7),  # ~15 GB
            dict(box_factor=5, points_per_waist=5, time_window_factor=6),      # ~11 GB: 16 GB cards
            dict(box_factor=4.5, points_per_waist=4.5, time_window_factor=6)]) # ~6.5 GB
grid, preset = pick_grid(cell, PRESETS, device_budget_gb(), n_fields=N_FIELDS, lambda0=LAM_GRID,
                         lambda_min=P["lambda_min"], lambda_max=P["lambda_max"], pulse_fwhm=P["window_fwhm"])
print(grid.describe())
print(f"grid carrier {grid.lambda0*1e9:.0f} nm, pulse centred at {P['lambda0']*1e9:.0f} nm")

OMEGA_C = 2 * np.pi * th.C0 / P["lambda0"]    # spectral phases below are expanded about the pulse centre


def gdd_phase(grid, gdd_fs2=0.0, tod_fs3=0.0):
    """GDD [fs^2] and TOD [fs^3] about the pulse centre (800 nm), not the grid carrier."""
    W = (grid.omega - OMEGA_C) * FS
    return (gdd_fs2 / 2 * W ** 2 + tod_fs3 / 6 * W ** 3) * grid.valid


def at_grid(w_800):
    """Beam radius given at 800 nm -> radius at the grid carrier (eigenmode scaling, w ~ sqrt(lambda))."""
    return w_800 * np.sqrt(grid.lambda0 / P["lambda0"])

## 4. Input pulse and injection
Ti:Sa spectra are not Gaussian. If you have the laser spectrum, put it in `data/daniault2024_input_spectrum.csv` (wavelength in nm, intensity) and it is used with a flat phase.

In [ ]:
meas_in = load_digitized(P["input_spectrum_csv"])
if meas_in is not None:
    E = th.measured_spectrum(grid, meas_in[0], meas_in[1])
    src = "measured spectrum, flat phase"
else:
    E = th.gaussian_spectrum(grid, fwhm_fs=P["fwhm"] / FS, center_nm=P["lambda0"] * 1e9)
    src = f"Gaussian, {P['fwhm']/FS:.0f} fs at {P['lambda0']*1e9:.0f} nm"

probe = th.Pulse.at_focus(grid, E, th.Beam(th.gaussian(at_grid(lin["w0"]))), P["energy"])
try:                                           # at 800 nm explicitly (cell.match would use the grid carrier)
    mm = cell.nonlinear_mode(probe.peak_power(), P["lambda0"], verbose=True)
except ValueError as err:                      # P >= Pc: no Kerr-matched mode
    print(err); mm = None
w0_ref = lin["w0"] if (P["injection"] == "linear" or mm is None) else mm["w0"]     # at 800 nm
pulse = th.Pulse.at_focus(grid, E, th.Beam(th.gaussian(at_grid(w0_ref)), chromatic_size="eigenmode"),
                          P["energy"])
t, Pt = integrated_power(grid, beam_pixels(pulse.Aw))
print(f"input: {src} | {fwhm(t, Pt):.1f} fs, peak {Pt.max()/1e9:.0f} GW, "
      f"P/Pc = {Pt.max()/gas.critical_power(P['lambda0']):.2f}, injected waist {w0_ref*1e6:.0f} um ({P['injection']})")

## 5. Run

In [ ]:
sim = th.MPCSimulation(pulse, cell, phi_max=PHI_MAX, mode_reference=at_grid(w0_ref), near_field=True,
                       record_every=1 if QUICK else RECORD_EVERY, cache_size=CACHE_SIZE)

In [ ]:
tag = run_tag(P, grid)
ckpt = os.path.join(RUN_DIR, f"checkpoint_{tag}.pkl")
if RESUME and os.path.exists(ckpt):
    sim.resume(ckpt)
    print(f"resumed from {ckpt} at pass {sim.passes_done}")
if not sim.done:
    t0 = time.time()
    sim.run(max_seconds=MAX_SECONDS)
    sim.checkpoint(ckpt)
    print(f"{time.time() - t0:.0f} s for this call")
print("finished" if sim.done else "time budget reached: run this cell again to continue")

## 6. Evolution along the cell
The last panel is the peak intensity at the cell centre. The barrier-suppression intensity of helium (≈ 1.5×10¹⁵ W/cm²) is drawn as a reference: ionisation is not in the model, so the run is only trustworthy well below it.

In [ ]:
h = sim.history
npass = np.array(h["pass"])
I_peak = 2 * np.array(h["peak_power"]) / (np.pi * np.array(h["w_peak_x"]) ** 2) * 1e-4   # W/cm^2
fig, ax = plt.subplots(1, 5, figsize=(18, 3.2), layout="constrained")
ax[0].plot(npass, np.array(h["energy"]) / h["energy"][0], "o-"); ax[0].set_title("transmission")
ax[1].plot(npass, h["B"], "o-"); ax[1].set_title("B-integral, on-axis peak (rad)")
ax[2].plot(npass, np.array(h["w_peak_x"]) * 1e6, "o-", label="x")
ax[2].axhline(w0_ref * 1e6, color="k", ls=":", label="injected waist")
ax[2].set_title("beam radius at the cell centre (um)"); ax[2].legend(fontsize=8)
ax[3].plot(npass, h["eta00_total"], "o-", label="HG00 fraction")
ax[3].plot(npass, h["V_near"], "s-", label="homogeneity (2D, near field)")
ax[3].plot(npass, h["M2x_mean"], "^-", label="M2 x"); ax[3].legend(fontsize=8)
ax[4].semilogy(npass, I_peak, "o-"); ax[4].axhline(1.5e15, color="r", ls="--", label="He barrier suppression")
ax[4].set_title("peak intensity (W/cm$^2$)"); ax[4].legend(fontsize=8)
for a in ax:
    a.set_xlabel("pass")
print(f"max peak intensity: {I_peak.max():.2e} W/cm^2 (pass {npass[np.argmax(I_peak)]})")

## 7. Output spectrum
Optional overlay: digitise the measured spectrum of Fig. 3 into `data/daniault2024_spectrum.csv` (nm, linear intensity). The RMS width and Fourier limit computed here are the quantities of the paper's Fig. 2.

In [ ]:
XLIM_NM = (450, 1150)
lam, S = dg.total_spectrum(grid, sim.Aw)
S_in = sim.history["spectrum"][0]
meas = load_digitized("data/daniault2024_spectrum.csv")

lam_c = np.sum(lam * S) / np.sum(S)
rms_nm = np.sqrt(np.sum((lam - lam_c) ** 2 * S) / np.sum(S)) * 1e9

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
for a, log in zip(ax, (False, True)):
    a.plot(sim.lam * 1e9, S_in / S_in.max(), "k:", label="input")
    a.plot(lam * 1e9, S / S.max(), label="simulation")
    if meas is not None:
        a.plot(*meas, label="measured (digitised)")
    a.set_xlim(*XLIM_NM); a.set_xlabel("wavelength (nm)")
    if log:
        a.set_yscale("log"); a.set_ylim(1e-3, 2)
ax[0].legend(fontsize=8)
print(f"RMS spectral width: {rms_nm:.0f} nm (centre of mass {lam_c*1e9:.0f} nm)")

## 8. Compression as in the experiment
The chirped mirrors are fixed (−640 fs²); the fused-silica thickness is scanned, as the wedges were in the experiment. `L_fit` is the total glass the simulated pulse needs on top of the mirrors. The real path also contained glass that is not known (exit window, quarter-wave plate), so compare `L_fit` with that total, not with zero.

In [ ]:
def n_fs(lam_um):
    """Fused silica, Malitson 1965."""
    l2 = lam_um ** 2
    return np.sqrt(1 + 0.6961663 * l2 / (l2 - 0.0684043 ** 2) + 0.4079426 * l2 / (l2 - 0.1162414 ** 2)
                   + 0.8974794 * l2 / (l2 - 9.896161 ** 2))

w = np.where(grid.valid, grid.omega, grid.omega0)
phi_mm = n_fs(2 * np.pi * th.C0 / w * 1e6) * w / th.C0 * 1e-3       # phase of 1 mm of fused silica
Wc = grid.omega - OMEGA_C
sel = grid.valid & (np.abs(Wc) < 0.1 * OMEGA_C)
c1, c0 = np.polyfit(Wc[sel], phi_mm[sel], 1)
phi_mm = (phi_mm - c1 * Wc - c0) * grid.valid                       # remove delay and constant (about 800 nm)

Epix = beam_pixels(sim.Aw)
UP = 8
phi_cm = gdd_phase(grid, P["cm_gdd"])
Ls = np.linspace(*P["fs_scan_mm"], 81)
peaks = [integrated_power(grid, Epix, phi_cm + L * phi_mm, UP)[1].max() for L in Ls]
L_fit = Ls[int(np.argmax(peaks))]
fine = np.linspace(L_fit - 0.1, L_fit + 0.1, 41)
L_fit = fine[int(np.argmax([integrated_power(grid, Epix, phi_cm + L * phi_mm, UP)[1].max() for L in fine]))]

t_tl, P_tl = fourier_limit(grid, Epix, UP)
t_e, P_e = integrated_power(grid, Epix, phi_cm + L_fit * phi_mm, UP)
gdds = np.linspace(-1500, 500, 81)                                # best pure GDD (about 800 nm)
g_best = gdds[int(np.argmax([integrated_power(grid, Epix, gdd_phase(grid, g), UP)[1].max() for g in gdds]))]
t_b, P_b = integrated_power(grid, Epix, gdd_phase(grid, g_best), UP)
best = {"gdd_fs2": g_best}
strehl = P_e.max() / P_tl.max()
centre = lambda t, Pw: t - t[np.argmax(Pw)]                        # plot every pulse centred on its peak

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
ax[0].plot(Ls, np.array(peaks) / P_tl.max(), "o-", ms=3)
ax[0].axvline(L_fit, color="k", ls=":"); ax[0].set_xlabel("fused silica (mm)")
ax[0].set_ylabel("peak / Fourier limit"); ax[0].set_title("-640 fs$^2$ + fused silica")
ax[1].plot(centre(t_tl, P_tl), P_tl / P_tl.max(), "k:", label=f"Fourier limit {fwhm(t_tl, P_tl):.2f} fs")
ax[1].plot(centre(t_e, P_e), P_e / P_tl.max(), label=f"-640 fs$^2$ + {L_fit:.2f} mm FS: {fwhm(t_e, P_e):.2f} fs, Strehl {strehl*100:.0f} %")
ax[1].plot(centre(t_b, P_b), P_b / P_tl.max(), "--", label=f"best pure GDD {best['gdd_fs2']:.0f} fs$^2$: {fwhm(t_b, P_b):.2f} fs")
ax[1].set_xlim(-25, 25); ax[1].set_xlabel("time (fs)"); ax[1].legend(fontsize=8)

## 9. Output beam: spatio-spectral homogeneity
The paper quotes the homogeneity at the FWHM and at the 1/e² points of the beam, so these are computed too.

In [ ]:
# near field (collimated output beam), along x: what the papers' imaging spectrometers measure
theta, lam_t, S_th = dg.spectrogram_theta_lambda(grid, sim.Aw, per_lambda=False)
V_x, V_mean = homogeneity_1d(S_th)
prof = S_th.sum(axis=1)
inside = prof > np.exp(-2) * prof.max()

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
ax[0].pcolormesh(lam_t * 1e9, theta * 1e3, S_th / S_th.max(), shading="auto", cmap="inferno")
ax[0].set_xlim(*XLIM_NM); ax[0].set_ylim(theta[inside].min() * 2e3, theta[inside].max() * 2e3)
ax[0].set_xlabel("wavelength (nm)"); ax[0].set_ylabel("angle (mrad)"); ax[0].set_title("output S(theta_x, lambda)")
ax[1].plot(theta * 1e3, prof / prof.max(), "k", label="profile")
ax[1].plot(theta[inside] * 1e3, V_x[inside], "g", label="homogeneity V")
ax[1].set_xlim(ax[0].get_ylim()); ax[1].set_ylim(0, 1.05); ax[1].legend(fontsize=8)
ax[1].set_xlabel("angle (mrad)"); ax[1].set_title(f"weighted homogeneity (x): {V_mean*100:.1f} %")

def v_within(level):
    ok = prof >= level * prof.max()
    return float(np.nanmin(V_x[ok]))
V_fwhm, V_e2 = v_within(0.5), v_within(np.exp(-2))
print(f"V >= {V_fwhm*100:.1f} % within the FWHM, >= {V_e2*100:.1f} % within 1/e^2 (paper: > 99 %, > 97 %)")

## 10. Summary

In [ ]:
Tcell = sim.history["energy"][-1] / sim.history["energy"][0]
table([
    ("waist / mirror spot (um, 1/e^2 radius)", f"{lin['w0']*1e6:.0f} / {lin['w_mirror']*1e6:.0f}", f"{REF['w0_um']} / ~{REF['wm_um']}"),
    ("transmission of the cell",               f"{Tcell*100:.1f} %", f"{REF['T']*100:.0f} %"),
    ("RMS spectral width (nm)",                f"{rms_nm:.0f}", "Fig. 2"),
    ("Fourier limit (fs)",                     f"{fwhm(t_tl, P_tl):.2f}", f"{REF['tl_fs']}"),
    ("compressed, -640 fs^2 + FS (fs)",        f"{fwhm(t_e, P_e):.2f}", f"{REF['comp_fs']}"),
    ("temporal Strehl ratio",                  f"{strehl*100:.0f} %", f"{REF['strehl']*100:.0f} %"),
    ("fused silica needed (mm)",               f"{L_fit:.2f}", "wedges, unknown"),
    ("M2 x / y (spectrally weighted)",         f"{sim.history['M2x_mean'][-1]:.2f} / {sim.history['M2y_mean'][-1]:.2f}", "-"),
    ("homogeneity FWHM / 1/e^2 (x)",           f"{V_fwhm*100:.1f} / {V_e2*100:.1f} %", REF["V"]),
    ("max peak intensity (W/cm^2)",            f"{I_peak.max():.1e}", "ionisation negligible"),
    ("B-integral, on-axis peak (rad)",         f"{sim.solver.B:.0f}", "-"),
])
sim.save(os.path.join(RUN_DIR, "run.npz"), include_field=False)
summary = dict(pressure_mbar=P["pressure_bar"] * 1000, rms_nm=rms_nm, ftl_fs=fwhm(t_tl, P_tl),
               comp_fs=fwhm(t_e, P_e), strehl=strehl, L_fit_mm=L_fit, T=Tcell, B=sim.solver.B,
               V_fwhm=V_fwhm, V_e2=V_e2, I_max=float(I_peak.max()), injection=P["injection"],
               grid=str(grid.shape), quick=QUICK)
with open(os.path.join(RUN_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=1)
print("saved", os.path.join(RUN_DIR, "summary.json"))

## 11. Pressure scan (paper Fig. 2)

Run this notebook at several pressures, one after the other (change `pressure_bar` in section 1 and restart the kernel; each pressure gets its own folder). A useful set: 0.4, 0.8, 1.2, 1.5, 1.8, 2.0, 2.2 bar; start with 1.8, then 0.8 and 2.2, which bracket the linear, saturated and distorted regimes. Long runs can be split with `MAX_SECONDS`. This cell collects every finished run.

The paper's behaviour to look for: RMS width growing linearly with pressure up to ~1200 mbar, then saturating; spectral modulations fading above 1200 mbar; a distorted, peaked spectrum and poor compressibility above 1800 mbar. To overlay the measurement, digitise Fig. 2 into `data/daniault2024_fig2_rms.csv` and `data/daniault2024_fig2_ftl.csv` (pressure in mbar, value).

In [ ]:
import glob
runs = []
for fn in sorted(glob.glob(os.path.join(SCAN_DIR, "*", "summary.json"))):
    with open(fn) as f:
        s = json.load(f)
    if not s.get("quick", False):
        runs.append(s)
runs.sort(key=lambda s: s["pressure_mbar"])
print(f"{len(runs)} finished runs:", [int(s["pressure_mbar"]) for s in runs])

if runs:
    p = np.array([s["pressure_mbar"] for s in runs])
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.4), layout="constrained")
    ax[0].plot(p, [s["rms_nm"] for s in runs], "o-", label="simulation"); ax[0].set_ylabel("RMS width (nm)")
    ax[1].plot(p, [s["ftl_fs"] for s in runs], "o-", label="Fourier limit")
    ax[1].plot(p, [s["comp_fs"] for s in runs], "s--", label="compressed (-640 fs$^2$ + FS)"); ax[1].set_ylabel("duration (fs)")
    ax[2].plot(p, [s["strehl"] for s in runs], "o-", label="Strehl")
    ax[2].plot(p, [s["V_e2"] for s in runs], "s-", label="homogeneity at 1/e$^2$"); ax[2].set_ylim(0, 1.05)
    for a, f in ((ax[0], "data/daniault2024_fig2_rms.csv"), (ax[1], "data/daniault2024_fig2_ftl.csv")):
        if os.path.exists(f):
            d = np.loadtxt(f, delimiter=","); d = d[np.argsort(d[:, 0])]
            a.plot(d[:, 0], d[:, 1], "k^", label="measured (digitised)")
    for a in ax:
        a.set_xlabel("He pressure (mbar)"); a.legend(fontsize=8)

## Checks before trusting the comparison
* **Convergence matters most here:** with a 1.5-cycle output, check `PHI_MAX = 0.01` and a finer preset at the operating pressure, and check that the spectrum stays well inside the spectral window (the solver tapers the nonlinear source at its edges).
* **n₂ of helium:** literature values scatter by ~15 %, and the scan is the best way to calibrate it: the pressure at which broadening saturates depends on n₂, dispersion and duration together, not on n₂ alone.
* **Injection:** at P/Pc ≈ 0.74, the Kerr lens is strong. Compare `injection="linear"` with `"nonlinear"`, and watch the peak-intensity plot.
* **Mirrors:** the enhanced-silver GDD and reflectivity versus wavelength are assumptions; the blue edge of the spectrum is sensitive to them.
* **Input:** the measured Ti:Sa spectrum (`data/`) instead of a Gaussian changes the shape of the output spectrum; the residual high-order phase of the real laser is not modelled.